In [ ]:
!pip install flash-attn --no-build-isolation -q
!pip install pathway sentence-transformers transformers torch -q

In [1]:
# ============================================================================
# NOTEBOOK 3: PATHWAY RATIONALE GENERATOR - FINAL
# Using SAME syntax and patterns from Notebook 2 (WORKING)
# ============================================================================

import pathway as pw
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import json
from datetime import datetime
from tqdm.auto import tqdm
import gc
import os

2026-01-11 17:37:38.802745: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768153059.142432     305 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768153059.245747     305 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768153060.048664     305 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768153060.048697     305 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768153060.048699     305 computation_placer.cc:177] computation placer alr

In [55]:
EMBEDDING_CONFIG = {
    'model_name': 'Alibaba-NLP/gte-Qwen2-7B-instruct',
    'dimension': 3584,
    'batch_size': 4,
    'trust_remote_code': True,
    'normalize': True,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

DATASET_NAME = '/kaggle/input/kdsh26-pathway-vector-embeddings-books-qwen'
DATASET_BASE = f'{DATASET_NAME}/kaggle/working/pathway_storage'
METADATA_PATH = f"{DATASET_BASE}/vector_store_metadata.json"

ENSEMBLE_CSV = '/kaggle/input/submission-ensemble-qwen-deberta-inference/Submission_Ensemble_Qwen_Deberta_Inference.csv'
TEST_CSV = '/kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset/test.csv'
OUTPUT_CSV = '/kaggle/working/predictions_with_rationales.csv'
SUBMISSION_CSV = '/kaggle/working/submission.csv'

device = EMBEDDING_CONFIG['device']

# Load and fix metadata
with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)

# Fix paths
for book_name in metadata['books'].keys():
    metadata['books'][book_name]['embeddings_npy'] = metadata['books'][book_name]['embeddings_npy'].replace(
        '/kaggle/working/pathway_storage', DATASET_BASE
    )
    metadata['books'][book_name]['chunks_csv'] = metadata['books'][book_name]['chunks_csv'].replace(
        '/kaggle/working/pathway_storage', DATASET_BASE
    )

print("✅ Configuration and metadata loaded")


✅ Configuration and metadata loaded


In [56]:
# ============================================================================
# STEP 1: Reload Correct Ensemble Predictions
# ============================================================================

print("\n" + "=" * 100)
print("🔄 LOADING CORRECT ENSEMBLE PREDICTIONS")
print("=" * 100)

# Load the CORRECT ensemble file
ENSEMBLE_CSV = '/kaggle/input/submission-ensemble-qwen-deberta-inference/Submission_Ensemble_Qwen_Deberta_Inference.csv'
ensemble_df = pd.read_csv(ENSEMBLE_CSV)

print(f"✅ Loaded ensemble predictions: {len(ensemble_df)} rows")
print(f"   Columns: {list(ensemble_df.columns)}")

# Check label distribution
print(f"\n📊 Label Distribution:")
print(ensemble_df['label'].value_counts())

# Convert text labels to numeric if needed
if ensemble_df['label'].dtype == 'object':
    label_map = {'consistent': 0, 'contradict': 1}
    ensemble_df['label_numeric'] = ensemble_df['label'].map(label_map)
    print(f"\n✅ Converted text labels to numeric")
else:
    ensemble_df['label_numeric'] = ensemble_df['label']

# Load test data
TEST_CSV = '/kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset/test.csv'
test_df = pd.read_csv(TEST_CSV)
print(f"\n✅ Loaded test data: {len(test_df)} rows")

# Fix column name
if 'book_name' in test_df.columns:
    test_df = test_df.rename(columns={'book_name': 'bookname'})
    print(f"   ✅ Renamed 'book_name' → 'bookname'")

# Merge properly
merged_df = test_df.merge(ensemble_df, on='id', how='inner')
print(f"\n✅ Merged: {len(merged_df)} rows")
print(f"   Columns: {list(merged_df.columns)}")

# Show sample to verify
print(f"\n📋 Sample Merged Data:")
sample_cols = ['id', 'bookname', 'char', 'label']
print(merged_df[sample_cols].head(5))



🔄 LOADING CORRECT ENSEMBLE PREDICTIONS
✅ Loaded ensemble predictions: 60 rows
   Columns: ['id', 'label']

📊 Label Distribution:
label
consistent    50
contradict    10
Name: count, dtype: int64

✅ Converted text labels to numeric

✅ Loaded test data: 60 rows
   ✅ Renamed 'book_name' → 'bookname'

✅ Merged: 60 rows
   Columns: ['id', 'bookname', 'char', 'caption', 'content', 'label', 'label_numeric']

📋 Sample Merged Data:
    id                    bookname      char       label
0   95   The Count of Monte Cristo  Noirtier  contradict
1  136   The Count of Monte Cristo     Faria  consistent
2   59  In Search of the Castaways  Thalcave  consistent
3   60  In Search of the Castaways  Thalcave  consistent
4  124   The Count of Monte Cristo     Faria  contradict


In [57]:
# ============================================================================
# STEP 2: Load Metadata - SAME AS NOTEBOOK 2
# ============================================================================

print(f"\n📁 Loading metadata from: {METADATA_PATH}")

with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)

print(f"✅ Metadata loaded")
print(f"   Books: {list(metadata['books'].keys())}")

# Fix paths - SAME AS NOTEBOOK 2
print("\n🔧 Fixing paths...")

for book_name in metadata['books'].keys():
    metadata['books'][book_name]['embeddings_npy'] = metadata['books'][book_name]['embeddings_npy'].replace(
        '/kaggle/working/pathway_storage',
        DATASET_BASE
    )
    metadata['books'][book_name]['chunks_csv'] = metadata['books'][book_name]['chunks_csv'].replace(
        '/kaggle/working/pathway_storage',
        DATASET_BASE
    )

if 'pathway_tables' in metadata:
    for book_name in metadata['pathway_tables'].keys():
        metadata['pathway_tables'][book_name]['parquet_path'] = metadata['pathway_tables'][book_name]['parquet_path'].replace(
            '/kaggle/working/pathway_storage',
            DATASET_BASE
        )

print("✅ Paths fixed!")



📁 Loading metadata from: /kaggle/input/kdsh26-pathway-vector-embeddings-books-qwen/kaggle/working/pathway_storage/vector_store_metadata.json
✅ Metadata loaded
   Books: ['In-search-of-the-castaways', 'The-count-of-monte-cristo']

🔧 Fixing paths...
✅ Paths fixed!


In [58]:
# ============================================================================
# STEP 3: Schemas - SAME AS NOTEBOOK 2
# ============================================================================

class ChunkWithEmbeddingSchema(pw.Schema):
    chunk_id: int
    book: str
    chunk: str
    chunk_length: int
    embedding: list
    embedding_dim: int
    score: float
    created_at: str

print("✅ Schemas defined")


✅ Schemas defined


In [59]:
# ============================================================================
# FIX: Update Book Name Normalization
# ============================================================================

def normalize_book_name(book_name):
    """
    Normalize book names to match embedding keys.
    Converts: "The Count of Monte Cristo" -> "The-count-of-monte-cristo"
    Keep first letter capitalized!
    """
    if pd.isna(book_name):
        return None
    
    # Strip whitespace
    normalized = book_name.strip()
    
    # Replace spaces with hyphens
    normalized = normalized.replace(' ', '-')
    
    # Convert to lowercase EXCEPT first letter of each word
    parts = normalized.split('-')
    # Capitalize only the first word, rest lowercase
    if parts:
        parts[0] = parts[0].capitalize()
        parts[1:] = [p.lower() for p in parts[1:]]
    normalized = '-'.join(parts)
    
    return normalized

print("✅ Updated normalization function")

# Test it
test_names = [
    "The Count of Monte Cristo",
    "In Search of the Castaways"
]

print("\nTesting normalization:")
for name in test_names:
    normalized = normalize_book_name(name)
    print(f"  '{name}' -> '{normalized}'")


✅ Updated normalization function

Testing normalization:
  'The Count of Monte Cristo' -> 'The-count-of-monte-cristo'
  'In Search of the Castaways' -> 'In-search-of-the-castaways'


In [60]:
# ============================================================================
# STEP 5: Rationale Generator with Book Evidence (FIXED)
# ============================================================================

class PathwayRationaleGenerator:
    """
    Generate rationales using book evidence from embeddings.
    """
    
    def __init__(self, metadata):
        self.metadata = metadata
        self.device = device
        self.embeddings = {}
        self.chunks_df = {}
        
        print("Loading vector stores...")
        self._load_stores()
        
        print(f"\nLoading {EMBEDDING_CONFIG['model_name']}...")
        self.embedding_model = SentenceTransformer(
            EMBEDDING_CONFIG['model_name'],
            device=device,
            trust_remote_code=True,
            config_kwargs={"use_cache": False}
        )
        print(f"✅ Embedding model loaded!")
    
    def _load_stores(self):
        """Load vector stores."""
        for book_name, book_meta in self.metadata['books'].items():
            emb_path = book_meta['embeddings_npy']
            chunks_path = book_meta['chunks_csv']
            
            # Load embeddings and chunks
            embeddings = np.load(emb_path)
            chunks = pd.read_csv(chunks_path)
            
            self.embeddings[book_name] = embeddings
            self.chunks_df[book_name] = chunks
            
            print(f"  ✓ {book_name}: {len(chunks)} chunks, {embeddings.shape}")
    
    def _retrieve_context(self, query: str, book_name: str, top_k: int = 3):
        """
        Retrieve relevant passages from book using embeddings.
        """
        # NORMALIZE BOOK NAME
        original_book_name = book_name
        book_name = normalize_book_name(book_name)
        
        if not query or not book_name:
            return []
        
        # Check if book exists
        if book_name not in self.embeddings:
            print(f"⚠️  Book '{original_book_name}' (normalized: '{book_name}') not found.")
            print(f"   Available: {list(self.embeddings.keys())}")
            return []
        
        try:
            # Encode query
            query_emb = self.embedding_model.encode([query], show_progress_bar=False)[0]
            
            # Get book data
            book_embs = self.embeddings[book_name]
            book_chunks = self.chunks_df[book_name]
            
            # Compute similarities
            similarities = np.dot(book_embs, query_emb)
            top_indices = np.argsort(similarities)[-top_k:][::-1]
            
            results = []
            for idx in top_indices:
                results.append({
                    'chunk': book_chunks.iloc[idx]['chunk'],
                    'score': float(similarities[idx]),
                    'book': book_name
                })
            
            return results
            
        except Exception as e:
            print(f"Error retrieving context: {str(e)}")
            return []
    
    def generate_rationale_with_evidence(self, test_row, prediction_label):
        """
        Generate rationale using retrieved book evidence.
        """
        char = test_row.get('char', 'Unknown')
        
        # FIX: Use 'bookname' if it exists, otherwise 'book'
        book_name = test_row.get('bookname') or test_row.get('book', 'Unknown')
        
        content = test_row.get('content', '')
        
        # Create query
        query = f"{char}: {content}"
        
        # Retrieve book passages
        passages = self._retrieve_context(query, book_name, top_k=3)
        
        if not passages or len(passages) == 0:
            return f"The statement is classified as {prediction_label}. No direct evidence found in the book."
        
        # Build rationale
        rationale_parts = []
        
        # Main evidence
        best_passage = passages[0]
        evidence = best_passage['chunk'][:200].replace('\n', ' ').strip()
        rationale_parts.append(
            f"The statement is {prediction_label} with the book. "
            f"Evidence (relevance: {best_passage['score']:.3f}): \"{evidence}...\""
        )
        
        # Additional supporting passages if high relevance
        if len(passages) > 1 and passages[1]['score'] > 0.5:
            second_evidence = passages[1]['chunk'][:150].replace('\n', ' ').strip()
            rationale_parts.append(
                f"Additional passage (score: {passages[1]['score']:.3f}): \"{second_evidence}...\""
            )
        
        return " ".join(rationale_parts)

print("✅ PathwayRationaleGenerator class defined")


✅ PathwayRationaleGenerator class defined


In [15]:
# ============================================================================
# STEP 4: Initialize Generator
# ============================================================================

print("\n" + "=" * 100)
print("🚀 INITIALIZING PATHWAY RATIONALE GENERATOR")
print("=" * 100)

generator = PathwayRationaleGenerator(metadata)



🚀 INITIALIZING PATHWAY RATIONALE GENERATOR
Loading vector stores...
  ✓ In-search-of-the-castaways: 375 chunks, (375, 3584)
  ✓ The-count-of-monte-cristo: 1290 chunks, (1290, 3584)

Loading Alibaba-NLP/gte-Qwen2-7B-instruct...


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Embedding model loaded!


In [61]:
# ============================================================================
# FIX: Rename book_name to bookname
# ============================================================================

print("\n" + "=" * 100)
print("🔧 FIXING COLUMN NAMES")
print("=" * 100)

if 'book_name' in merged_df.columns:
    # FIX: Change 'book_name' to 'bookname' (not 'book_name')
    merged_df = merged_df.rename(columns={'book_name': 'bookname'})
    print("✅ Renamed 'book_name' -> 'bookname'")
    print(f"   Updated columns: {list(merged_df.columns)}")
else:
    print("❌ ERROR: 'book_name' column not found!")

# Verify fix worked
if 'bookname' in merged_df.columns:
    print("\n✅ Column fix successful!")
    print(f"\n📚 Books in dataset:")
    # FIX: Use 'bookname' not 'book_name'
    for book, count in merged_df['bookname'].value_counts().items():
        print(f"   - {book}: {count} statements")
else:
    print("\n❌ Fix failed - bookname column still missing")



🔧 FIXING COLUMN NAMES
❌ ERROR: 'book_name' column not found!

✅ Column fix successful!

📚 Books in dataset:
   - In Search of the Castaways: 37 statements
   - The Count of Monte Cristo: 23 statements


In [32]:
# ============================================================================
# STEP 5: Load Data (FIXED)
# ============================================================================

print("\n" + "=" * 100)
print("📂 LOADING DATA")
print("=" * 100)

# Load test data
print("\nLoading test data...")
test_df = pd.read_csv(TEST_CSV)
print(f"✅ Loaded {len(test_df)} test examples")
print(f"   Columns: {list(test_df.columns)}")

# Standardize column names in test data
if 'book' in test_df.columns and 'bookname' not in test_df.columns:
    test_df = test_df.rename(columns={'book': 'bookname'})
    print("   📝 Renamed 'book' -> 'bookname'")

# Load ensemble predictions
print("\nLoading ensemble predictions...")
ensemble_df = pd.read_csv(ENSEMBLE_CSV)
print(f"✅ Loaded {len(ensemble_df)} predictions")
print(f"   Columns: {list(ensemble_df.columns)}")

# Merge test data with predictions
print("\nMerging test data with predictions...")
merged_df = test_df.merge(ensemble_df, on='id', how='inner')
print(f"✅ Merged {len(merged_df)} rows")
print(f"   Final columns: {list(merged_df.columns)}")

# Verify required columns
required_columns = ['id', 'book_name', 'char', 'content', 'label']
missing_columns = [col for col in required_columns if col not in merged_df.columns]

if missing_columns:
    print(f"\n❌ ERROR: Missing columns: {missing_columns}")
    print(f"\nAttempting to fix...")
    
    # Try common alternatives
    if 'book' in merged_df.columns and 'bookname' in missing_columns:
        merged_df = merged_df.rename(columns={'book': 'bookname'})
        print("   ✅ Fixed: 'book' -> 'bookname'")
else:
    print(f"\n✅ All required columns present!")
    
# Show sample
print(f"\n📋 Sample merged data:")
print(merged_df[['id', 'book_name', 'char', 'label']].head(3))
print(f"\n📚 Books in dataset:")
for book, count in merged_df['book_name'].value_counts().items():
    print(f"   - {book}: {count} statements")



📂 LOADING DATA

Loading test data...
✅ Loaded 60 test examples
   Columns: ['id', 'book_name', 'char', 'caption', 'content']

Loading ensemble predictions...
✅ Loaded 60 predictions
   Columns: ['id', 'label']

Merging test data with predictions...
✅ Merged 60 rows
   Final columns: ['id', 'book_name', 'char', 'caption', 'content', 'label']

✅ All required columns present!

📋 Sample merged data:
    id                   book_name      char       label
0   95   The Count of Monte Cristo  Noirtier  contradict
1  136   The Count of Monte Cristo     Faria  consistent
2   59  In Search of the Castaways  Thalcave  consistent

📚 Books in dataset:
   - In Search of the Castaways: 37 statements
   - The Count of Monte Cristo: 23 statements


In [36]:
# ============================================================================
# COMPLETE FIX: Reload Data with Correct Column Names
# ============================================================================

print("\n" + "=" * 100)
print("🔄 RELOADING DATA WITH PROPER FIX")
print("=" * 100)

# Load test data fresh
test_df = pd.read_csv(TEST_CSV)
print(f"✅ Loaded test data: {len(test_df)} rows")
print(f"   Original columns: {list(test_df.columns)}")

# Fix column name IMMEDIATELY
if 'book_name' in test_df.columns:
    test_df = test_df.rename(columns={'book_name': 'bookname'})
    print("   ✅ Renamed 'book_name' → 'bookname'")

# Load ensemble predictions
ensemble_df = pd.read_csv(ENSEMBLE_CSV)
print(f"✅ Loaded ensemble: {len(ensemble_df)} rows")

# Merge
merged_df = test_df.merge(ensemble_df, on='id', how='inner')
print(f"✅ Merged: {len(merged_df)} rows")
print(f"   Final columns: {list(merged_df.columns)}")

# Verify bookname exists
if 'bookname' in merged_df.columns:
    print("\n✅ SUCCESS: 'bookname' column exists!")
    print(f"\n📚 Books in dataset:")
    for book, count in merged_df['bookname'].value_counts().items():
        print(f"   - {book}: {count} statements")
else:
    print("\n❌ ERROR: 'bookname' still missing!")
    print(f"   Columns: {list(merged_df.columns)}")



🔄 RELOADING DATA WITH PROPER FIX
✅ Loaded test data: 60 rows
   Original columns: ['id', 'book_name', 'char', 'caption', 'content']
   ✅ Renamed 'book_name' → 'bookname'
✅ Loaded ensemble: 60 rows
✅ Merged: 60 rows
   Final columns: ['id', 'bookname', 'char', 'caption', 'content', 'label']

✅ SUCCESS: 'bookname' column exists!

📚 Books in dataset:
   - In Search of the Castaways: 37 statements
   - The Count of Monte Cristo: 23 statements


In [62]:
# ============================================================================
# STEP: Generate Rationales with CORRECT Labels
# ============================================================================

print("\n" + "=" * 100)
print("📝 GENERATING RATIONALES WITH CORRECT LABELS")
print("=" * 100)

print(f"\nGenerating rationales for {len(merged_df)} predictions...")

rationales = []

for idx, row in tqdm(merged_df.iterrows(), total=len(merged_df), desc="Processing"):
    # Use the CORRECT label from ensemble
    label_val = row['label']
    if isinstance(label_val, str):
        prediction_label = label_val.lower()
    else:
        prediction_label = "contradict" if label_val == 1 else "consistent"
    
    # Generate rationale
    rationale = generator.generate_rationale_with_evidence(row, prediction_label)
    rationales.append(rationale)

# Add rationales
merged_df['rationale'] = rationales

print(f"\n✅ All rationales generated!")

# ============================================================================
# STEP: Create Final Submission
# ============================================================================

print("\n" + "=" * 100)
print("📄 CREATING FINAL SUBMISSION")
print("=" * 100)

# Create submission
submission = pd.DataFrame({
    'Story ID': merged_df['id'],
    'Prediction': merged_df['label'],
    'Rationale': merged_df['rationale']
})

# Fix grammar
def fix_rationale_grammar(rationale, prediction):
    """Fix grammar to match prediction."""
    if prediction == 'contradict' or prediction == 1:
        rationale = rationale.replace('is contradict with the book', 'contradicts the novel')
        rationale = rationale.replace('The statement is consistent', 'The statement contradicts')
        if not rationale.startswith('The statement contradicts'):
            rationale = rationale.replace('The statement ', 'The statement contradicts the novel. ', 1)
    else:
        rationale = rationale.replace('is contradict with the book', 'is consistent with the novel')
        rationale = rationale.replace('The statement contradicts', 'The statement is consistent with')
    
    return rationale

submission['Rationale'] = submission.apply(
    lambda row: fix_rationale_grammar(row['Rationale'], row['Prediction']), 
    axis=1
)

# Save
FINAL_SUBMISSION = '/kaggle/working/submission_final.csv'
submission.to_csv(FINAL_SUBMISSION, index=False)

print(f"\n✅ Submission saved: {FINAL_SUBMISSION}")
print(f"📊 Columns: {list(submission.columns)}")
print(f"📈 Rows: {len(submission)}")

# Show prediction distribution
print(f"\n📊 Prediction Distribution:")
print(submission['Prediction'].value_counts())

# Show samples
print(f"\n{'='*100}")
print("📋 SAMPLE SUBMISSIONS")
print('='*100)

for i in range(min(5, len(submission))):
    print(f"\nID: {submission.iloc[i]['Story ID']} | Prediction: {submission.iloc[i]['Prediction']}")
    print(f"Rationale: {submission.iloc[i]['Rationale'][:200]}...")

print("\n✅ READY TO SUBMIT!")



📝 GENERATING RATIONALES WITH CORRECT LABELS

Generating rationales for 60 predictions...


Processing:   0%|          | 0/60 [00:00<?, ?it/s]


✅ All rationales generated!

📄 CREATING FINAL SUBMISSION

✅ Submission saved: /kaggle/working/submission_final.csv
📊 Columns: ['Story ID', 'Prediction', 'Rationale']
📈 Rows: 60

📊 Prediction Distribution:
Prediction
consistent    50
contradict    10
Name: count, dtype: int64

📋 SAMPLE SUBMISSIONS

ID: 95 | Prediction: contradict
Rationale: The statement contradicts the novel. Evidence (relevance: 0.603): "I know M. Franz d’Épinay,” said the count; “is he not the son of General de Quesnel, who was created Baron d’Épinay by Charles X.?” “...

ID: 136 | Prediction: consistent
Rationale: The statement is consistent with the book. Evidence (relevance: 0.490): "149’” “Well!” said Faria, when the young man had finished reading it. “Why,” replied Dantès, “I see nothing but broken lines an...

ID: 59 | Prediction: consistent
Rationale: The statement is consistent with the book. Evidence (relevance: 0.532): "he is mentioned in the accounts of the Indians. He was a brave man." "You understand, m

In [41]:
# ============================================================================
# DEBUG: Check what's actually in the row
# ============================================================================

print("\n" + "=" * 100)
print("🔍 DEBUGGING ROW DATA")
print("=" * 100)

# Check first row
sample_row = merged_df.iloc[0]
print(f"\nSample row columns: {list(sample_row.index)}")
print(f"\nSample row data:")
for col in sample_row.index:
    print(f"  {col}: {sample_row[col]}")

# Check if bookname exists
if 'bookname' in sample_row.index:
    print(f"\n✅ 'bookname' exists in row: {sample_row['bookname']}")
else:
    print(f"\n❌ 'bookname' does NOT exist in row")
    print(f"   Available columns: {list(sample_row.index)}")



🔍 DEBUGGING ROW DATA

Sample row columns: ['id', 'bookname', 'char', 'caption', 'content', 'label', 'rationale']

Sample row data:
  id: 95
  bookname: The Count of Monte Cristo
  char: Noirtier
  caption: The Fatal Decision of the Hundred Days
  content: Learning that Villefort meant to denounce him to Louis XVIII, Noirtier pre-emptively handed the conspiracy dossier to a British spy—the very file the Count of Monte Cristo later acquired—thereby engineering his son’s “lawful” murder.
  label: contradict
  rationale: The statement is contradict with the book. Evidence (relevance: 0.603): "I know M. Franz d’Épinay,” said the count; “is he not the son of General de Quesnel, who was created Baron d’Épinay by Charles X.?” “The same,” said Villefort. “Well, but he is a charming young man, a..." Additional passage (score: 0.587): "Louis XVIII. to break my vow in behalf of the ex-emperor.” This answer was too clear to permit of any mistake as to his sentiments. “General,” said th..."

✅ 'boo

In [44]:
print(merged_df)

     id                    bookname                  char  \
0    95   The Count of Monte Cristo              Noirtier   
1   136   The Count of Monte Cristo                 Faria   
2    59  In Search of the Castaways              Thalcave   
3    60  In Search of the Castaways              Thalcave   
4   124   The Count of Monte Cristo                 Faria   
5   111   The Count of Monte Cristo              Noirtier   
6   135   The Count of Monte Cristo                 Faria   
7    27  In Search of the Castaways  Tom Ayrton/Ben Joyce   
8   110   The Count of Monte Cristo              Noirtier   
9    42  In Search of the Castaways  Tom Ayrton/Ben Joyce   
10   56  In Search of the Castaways              Thalcave   
11   80  In Search of the Castaways            Kai-Koumou   
12   28  In Search of the Castaways  Tom Ayrton/Ben Joyce   
13  101   The Count of Monte Cristo              Noirtier   
14   91   The Count of Monte Cristo              Noirtier   
15   24  In Search of th

In [50]:
# ============================================================================
# FINAL STEP: Create Proper Submission File
# ============================================================================

print("\n" + "=" * 100)
print("📄 CREATING FINAL SUBMISSION")
print("=" * 100)

# Create properly formatted submission
submission = pd.DataFrame({
    'Story ID': merged_df['id'],
    'Prediction': merged_df['label'].apply(lambda x: 'contradict' if x == 1 else 'consistent'),
    'Rationale': merged_df['rationale']
})

# Fix grammar in rationales
def fix_rationale_grammar(rationale):
    """Fix common grammar errors in rationales."""
    # Fix "is contradict with" -> "contradicts"
    rationale = rationale.replace('is contradict with the book', 'contradicts the book')
    rationale = rationale.replace('is consistent with the book', 'is consistent with the book')
    
    # Ensure proper sentence structure
    if not rationale.endswith('.') and not rationale.endswith('"'):
        rationale += '.'
    
    return rationale

submission['Rationale'] = submission['Rationale'].apply(fix_rationale_grammar)

# Save final submission
FINAL_SUBMISSION = '/kaggle/working/submission_final.csv'
submission.to_csv(FINAL_SUBMISSION, index=False)

print(f"\n✅ Submission created: {FINAL_SUBMISSION}")
print(f"📊 Format: {list(submission.columns)}")
print(f"📈 Rows: {len(submission)}")
print(f"💾 Size: {os.path.getsize(FINAL_SUBMISSION) / 1024:.1f} KB")

# Quality check
print(f"\n📋 Prediction Distribution:")
print(submission['Prediction'].value_counts())

# Show samples
print(f"\n{'='*100}")
print("📋 SAMPLE HIGH-QUALITY SUBMISSIONS")
print('='*100)

for i in range(min(3, len(submission))):
    print(f"\n{'-'*100}")
    print(f"Story ID: {submission.iloc[i]['Story ID']}")
    print(f"Prediction: {submission.iloc[i]['Prediction']}")
    print(f"\nRationale:")
    print(submission.iloc[i]['Rationale'])

# Verify format
print(f"\n{'='*100}")
print("✅ SUBMISSION FORMAT VERIFICATION")
print('='*100)
print(f"✅ Column 1: Story ID (type: {submission['Story ID'].dtype})")
print(f"✅ Column 2: Prediction (values: {submission['Prediction'].unique()})")
print(f"✅ Column 3: Rationale (avg length: {submission['Rationale'].str.len().mean():.0f} chars)")

print("\n✅ Ready to submit!")



📄 CREATING FINAL SUBMISSION

✅ Submission created: /kaggle/working/submission_final.csv
📊 Format: ['Story ID', 'Prediction', 'Rationale']
📈 Rows: 60
💾 Size: 26.0 KB

📋 Prediction Distribution:
Prediction
consistent    60
Name: count, dtype: int64

📋 SAMPLE HIGH-QUALITY SUBMISSIONS

----------------------------------------------------------------------------------------------------
Story ID: 95
Prediction: consistent

Rationale:
The statement contradicts the book. Evidence (relevance: 0.603): "I know M. Franz d’Épinay,” said the count; “is he not the son of General de Quesnel, who was created Baron d’Épinay by Charles X.?” “The same,” said Villefort. “Well, but he is a charming young man, a..." Additional passage (score: 0.587): "Louis XVIII. to break my vow in behalf of the ex-emperor.” This answer was too clear to permit of any mistake as to his sentiments. “General,” said th..."

----------------------------------------------------------------------------------------------------
Sto

In [48]:
# # ============================================================================
# # STEP 8: Verification & Samples
# # ============================================================================

# print("\n" + "=" * 100)
# print("📊 VERIFICATION & SAMPLES")
# print("=" * 100)

# # Label distribution
# print("\n📊 Label distribution:")
# print(submission['label'].value_counts())

# # Sample rationales
# print("\n" + "=" * 100)
# print("📝 SAMPLE RATIONALES")
# print("=" * 100)

# for i in range(min(5, len(ensemble_d))):
#     row = ensemble_df.iloc[i]
    
#     print(f"\n{'='*100}")
#     print(f"Example {i+1} (ID: {row.get('id', '?')})")
#     print(f"{'='*100}")
#     print(f"Character: {row.get('char', 'N/A')}")
#     print(f"Book: {row.get('bookname', 'N/A')}")
#     print(f"Prediction: {submission.iloc[i]['label']}")
#     print(f"Confidence: {row.get('confidence', 'N/A')}")
#     print(f"\n📄 Statement:")
#     print(f"  {str(row.get('content', ''))[:250]}...")
#     print(f"\n💡 Rationale:")
#     print(f"  {row['rationale']}")
#     print(f"\n🔧 Technical:")
#     print(f"  {row['rationale_technical']}")



📊 VERIFICATION & SAMPLES

📊 Label distribution:
label
consistent    60
Name: count, dtype: int64

📝 SAMPLE RATIONALES

Example 1 (ID: 95)
Character: N/A
Book: N/A
Prediction: consistent
Confidence: N/A

📄 Statement:
  ...

💡 Rationale:


KeyError: 'rationale'

In [49]:
# ============================================================================
# STEP 9: Quality Metrics - FIXED for missing model columns
# ============================================================================

print("\n" + "=" * 100)
print("📈 QUALITY METRICS")
print("=" * 100)

# Rationale length statistics
rationale_lengths = ensemble_df['rationale'].str.len()

print(f"\n📏 Rationale length statistics:")
print(f"  Mean: {rationale_lengths.mean():.1f} characters")
print(f"  Median: {rationale_lengths.median():.1f} characters")
print(f"  Min: {rationale_lengths.min()} characters")
print(f"  Max: {rationale_lengths.max()} characters")

# Label consistency check
print(f"\n✅ Label consistency (rationale matches prediction):")
consistent_count = 0
contradict_count = 0

for idx, row in ensemble_df.iterrows():
    rationale_lower = str(row['rationale']).lower()
    predicted_label = 'contradict' if row['prediction'] == 1 else 'consistent'
    
    if predicted_label in rationale_lower:
        if predicted_label == 'consistent':
            consistent_count += 1
        else:
            contradict_count += 1

total_checked = len(ensemble_df)
consistency_rate = (consistent_count + contradict_count) / total_checked * 100

print(f"  Rationales mentioning 'consistent': {consistent_count}/{total_checked}")
print(f"  Rationales mentioning 'contradict': {contradict_count}/{total_checked}")
print(f"  Label consistency rate: {consistency_rate:.1f}%")

# Prediction distribution by book
print(f"\n📚 Predictions by book:")

if 'bookname' in ensemble_df.columns:
    for book in ensemble_df['bookname'].dropna().unique():
        book_df = ensemble_df[ensemble_df['bookname'] == book]
        if len(book_df) > 0:
            consistent = (book_df['prediction'] == 0).sum()
            contradict = (book_df['prediction'] == 1).sum()
            print(f"\n  📖 {book}:")
            print(f"     Consistent: {consistent} ({consistent/len(book_df)*100:.1f}%)")
            print(f"     Contradict: {contradict} ({contradict/len(book_df)*100:.1f}%)")
else:
    print("  ⚠️ 'bookname' column not found - skipping per-book analysis")

# ✅ FIX: Only compute model agreement if columns exist
model_cols_exist = all(col in ensemble_df.columns for col in ['qwen_v2_pred', 'deberta_pred', 'qwen_v1_pred'])

if model_cols_exist:
    print(f"\n🤝 Model agreement statistics:")
    ensemble_df['all_agree'] = (
        (ensemble_df['qwen_v2_pred'] == ensemble_df['deberta_pred']) & 
        (ensemble_df['deberta_pred'] == ensemble_df['qwen_v1_pred'])
    )
    ensemble_df['majority_agree'] = (
        ((ensemble_df['qwen_v2_pred'] + ensemble_df['deberta_pred'] + ensemble_df['qwen_v1_pred']) >= 2) |
        ((ensemble_df['qwen_v2_pred'] + ensemble_df['deberta_pred'] + ensemble_df['qwen_v1_pred']) <= 1)
    )
    
    all_agree_count = ensemble_df['all_agree'].sum()
    majority_agree_count = ensemble_df['majority_agree'].sum()
    
    print(f"  All 3 models agree: {all_agree_count}/{len(ensemble_df)} ({all_agree_count/len(ensemble_df)*100:.1f}%)")
    print(f"  Majority agreement (2+): {majority_agree_count}/{len(ensemble_df)} ({majority_agree_count/len(ensemble_df)*100:.1f}%)")
else:
    print(f"\n🤝 Model agreement statistics:")
    print(f"  ⚠️ Individual model columns not found - skipping agreement analysis")

# Confidence statistics
if 'confidence' in ensemble_df.columns:
    ensemble_df['confidence_float'] = ensemble_df['confidence'].astype(float)
    avg_confidence = ensemble_df['confidence_float'].mean()
    high_confidence = (ensemble_df['confidence_float'] >= 0.8).sum()
    low_confidence = (ensemble_df['confidence_float'] < 0.6).sum()
    
    print(f"\n📊 Confidence statistics:")
    print(f"  Average confidence: {avg_confidence:.4f}")
    print(f"  High confidence (≥0.8): {high_confidence}/{len(ensemble_df)} ({high_confidence/len(ensemble_df)*100:.1f}%)")
    print(f"  Low confidence (<0.6): {low_confidence}/{len(ensemble_df)} ({low_confidence/len(ensemble_df)*100:.1f}%)")
else:
    print(f"\n📊 Confidence statistics:")
    print(f"  ⚠️ 'confidence' column not found - skipping confidence analysis")
    avg_confidence = 0.5  # Default for later use



📈 QUALITY METRICS


KeyError: 'rationale'

In [ ]:
# ============================================================================
# STEP 10: Final Summary
# ============================================================================

print("\n" + "=" * 100)
print("✅ PIPELINE COMPLETE")
print("=" * 100)

print(f"\n📁 Files created:")
print(f"\n   1. {OUTPUT_CSV}")
print(f"      → Full predictions with rationales (for analysis)")
print(f"      → Columns: id, bookname, char, prediction, confidence, rationale, rationale_technical, etc.")
print(f"      → Rows: {len(ensemble_df)}")
print(f"      → Size: {os.path.getsize(OUTPUT_CSV) / 1024:.1f} KB")

print(f"\n   2. {SUBMISSION_CSV} ⭐")
print(f"      → Kaggle submission format")
print(f"      → Columns: id, label")
print(f"      → Rows: {len(submission)}")
print(f"      → Size: {os.path.getsize(SUBMISSION_CSV) / 1024:.1f} KB")
print(f"      → Ready to submit!")

print(f"\n📊 Final statistics:")
print(f"   Total predictions: {len(ensemble_df)}")
print(f"   Consistent: {(ensemble_df['label'] == 0).sum()} ({(ensemble_df['label'] == 0).sum()/len(ensemble_df)*100:.1f}%)")
print(f"   Contradict: {(ensemble_df['label'] == 1).sum()} ({(ensemble_df['label'] == 1).sum()/len(ensemble_df)*100:.1f}%)")
print(f"   Average confidence: {avg_confidence:.4f}")
print(f"   Rationales generated: {len(ensemble_df)}")
print(f"   With retrieved context: {len([r for r in rationales if len(r) > 50])}")

print("\n" + "=" * 100)
print("🎉 SUCCESS! Ready to submit to Kaggle!")
print("=" * 100)

print(f"\n📤 Next steps:")
print(f"   1. Download {SUBMISSION_CSV}")
print(f"   2. Go to Kaggle competition page")
print(f"   3. Click 'Submit Predictions'")
print(f"   4. Upload submission.csv")
print(f"   5. Submit and check leaderboard!")

print(f"\n💡 Optional analysis:")
print(f"   • Review {OUTPUT_CSV} for detailed rationales")
print(f"   • Check rationale quality and consistency")
print(f"   • Analyze model agreement patterns")


In [12]:
# ============================================================================
# STEP 11: Cleanup GPU Memory
# ============================================================================

print("\n" + "=" * 100)
print("🧹 CLEANUP")
print("=" * 100)

print("\n🧹 Cleaning up GPU memory...")

# Delete models
if hasattr(generator, 'embedding_model'):
    del generator.embedding_model
if hasattr(generator, 'generator'):
    del generator.generator

gc.collect()
torch.cuda.empty_cache()

final_vram_allocated = torch.cuda.memory_allocated() / 1024**3
final_vram_reserved = torch.cuda.memory_reserved() / 1024**3

print(f"✅ GPU memory cleaned")
print(f"   Allocated: {final_vram_allocated:.2f} GB")
print(f"   Reserved: {final_vram_reserved:.2f} GB")



🧹 CLEANUP

🧹 Cleaning up GPU memory...
✅ GPU memory cleaned
   Allocated: 0.03 GB
   Reserved: 0.03 GB


In [ ]:
# ============================================================================
# STEP 12: Final Validation
# ============================================================================

print("\n" + "=" * 100)
print("🔍 FINAL VALIDATION")
print("=" * 100)

# Validate submission format
print("\n📋 Submission file validation:")
print(f"   Columns: {list(submission.columns)}")
print(f"   Expected: ['id', 'label']")
print(f"   Match: {'✅' if list(submission.columns) == ['id', 'label'] else '❌'}")

# Check for nulls
print(f"\n   Null values:")
print(f"     id: {submission['id'].isna().sum()}")
print(f"     label: {submission['label'].isna().sum()}")

# Check label values
unique_labels = submission['label'].unique()
print(f"\n   Unique labels: {sorted(unique_labels)}")
print(f"   Expected: ['consistent', 'contradict']")
print(f"   Match: {'✅' if set(unique_labels).issubset({'consistent', 'contradict'}) else '❌'}")

# Sample submission rows
print(f"\n📄 Sample submission rows:")
print(submission.head(10))

print("\n" + "=" * 100)
print("✅ VALIDATION PASSED - READY TO SUBMIT!")
print("=" * 100)

print(f"\n🎯 Summary:")
print(f"   ✅ Vector stores loaded from gte-Qwen2-7B embeddings")
print(f"   ✅ Retrieved relevant book context for each prediction")
print(f"   ✅ Generated {len(ensemble_df)} rationales")
print(f"   ✅ Created submission.csv in correct format (id, label)")
print(f"   ✅ All validations passed")

print(f"\n🚀 You're ready to submit to Kaggle!")
print(f"\n💾 Download file: {SUBMISSION_CSV}")

print("\nALL DONE! Good luck!")
